In [ ]:
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from constants import UCI_CLIENT_SITES_TO_VALIDATE, UCI_VALIDATION_START, UCI_VALIDATION_WINDOW

# UCI Forecast Evaluation

This notebook evaluates load forecasts generated by seasonally naive, ETS and TCN models across all sites. It computes MAE, WAPE, and MSSE error metrics and summarises results.

In [ ]:
def mean_absolute_error(y_true: np.ndarray, y_hat: np.ndarray) -> float:
    return np.mean(np.abs(y_true - y_hat))


def weighted_absolute_pct_error(y_true: np.ndarray, y_hat: np.ndarray) -> float:
    return np.sum(np.abs(y_true - y_hat)) / np.sum(np.abs(y_true))


def mean_squared_scaled_error(y_true: np.ndarray, y_hat: np.ndarray, y_train: np.ndarray, period: int = 1) -> float:
    # Calculate the in-sample naive train error
    y_hat_naive = y_train[:-period]
    y_actual = y_train[period:]
    naive_errors = y_actual - y_hat_naive
    mse_naive = np.mean(naive_errors ** 2)
    
    # Calculate forecast errors
    mse_forecast = np.mean((y_true - y_hat) ** 2)
    
    return  mse_forecast / mse_naive

In [ ]:
# Constants

UCI_DATA_DIR = Path("../../data/uci")
UCI_DATA_FILE_NAME = "preprocessed.pq"
UCI_DATA_PATH = UCI_DATA_DIR / UCI_DATA_FILE_NAME

N_VALIDATION_FOLDS = 10
SEASONAL_PERIOD = 96

MODELS = ["naive", "ets", "tcn"]

RESULTS_OUTPUT_DIR = Path("../../results/uci")

In [ ]:
# Load preprocessed data

uci_df = pl.read_parquet(UCI_DATA_PATH)
UCI_CLIENT_DF = uci_df.filter(pl.col("client").is_in(UCI_CLIENT_SITES_TO_VALIDATE))

In [ ]:
# Compute errors for each site and model

site_model_errors = {}

for site in UCI_CLIENT_SITES_TO_VALIDATE:
    client_df = UCI_CLIENT_DF.filter(pl.col("client") == site)

    model_errors = defaultdict(list)
    for fold in range(N_VALIDATION_FOLDS):
        val_start = UCI_VALIDATION_START + fold * UCI_VALIDATION_WINDOW
        val_end = val_start + UCI_VALIDATION_WINDOW
        train_df = (
            client_df
            .filter(pl.col("timestamp").lt(val_start))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by=pl.col("timestamp"))
        )
        y_train = train_df["demand"].to_numpy()
    
        for model in MODELS:
            model_site_dir = RESULTS_OUTPUT_DIR / model / site
            forecasts_file = model_site_dir / f"forecasts_{site}_fold_{fold}.pq"
            
            forecasts_df = pl.read_parquet(forecasts_file)
            y_true = forecasts_df["demand"].to_numpy()
            y_hat = forecasts_df["forecast"].to_numpy()
            
            mae = mean_absolute_error(y_true, y_hat),
            wape = weighted_absolute_pct_error(y_true, y_hat)
            rmsse = np.sqrt(mean_squared_scaled_error(y_true, y_hat, y_train, period=SEASONAL_PERIOD))
            model_error_dict = {
                "fold_index": fold,
                "mean_absolute_error": mae,
                "weighted_absolute_pct_error": wape,
                "root_mean_squared_scaled_error": rmsse
            }
            model_errors[model].append(model_error_dict)

    site_model_errors[site] = model_errors

In [ ]:
# Create visualization of forecast errors by site and model

error_key = "weighted_absolute_pct_error"
error_label = "WAPE"

n_sites = len(UCI_CLIENT_SITES_TO_VALIDATE)
n_cols = 4
n_rows = (n_sites + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharex=True)
axes = axes.flatten() if n_sites > 1 else [axes]

# Color map for models
colors = plt.cm.tab10(np.linspace(0, 1, len(MODELS)))
model_colors = dict(zip(MODELS, colors))

for idx, site in enumerate(UCI_CLIENT_SITES_TO_VALIDATE):
    ax = axes[idx]
    
    # Plot each model's errors
    for model in MODELS:
        model_data = site_model_errors[site][model]
        fold_indices = [d["fold_index"] for d in model_data]
        rmsse_values = [d[error_key] for d in model_data]
        
        # Only add label for first subplot to avoid duplicate legends
        label = model.upper() if idx == 0 else None
        
        ax.plot(
            fold_indices,
            rmsse_values,
            marker='o',
            label=label, 
            color=model_colors[model],
            linewidth=2.5,
            markersize=7
        )
    
    ax.set_title(f'{site}', fontsize=12)
    ax.grid(True, alpha=0.3)

    if idx % n_cols == 0:
        ax.set_ylabel(error_label)

    if idx // n_cols == n_rows - 1:
        ax.set_xlabel("Validation Fold Index")

fig.legend(bbox_to_anchor=(0.93, 1.01), loc='center', ncol=len(MODELS))
fig.align_labels()
fig.tight_layout();

In [ ]:
# Calculate average and standard deviation of errors across folds
error_metrics = ["mean_absolute_error", "weighted_absolute_pct_error", "root_mean_squared_scaled_error"]

site_model_summary = {}
for site in UCI_CLIENT_SITES_TO_VALIDATE:
    site_model_summary[site] = {}
    
    for model in MODELS:
        model_data = site_model_errors[site][model]
        site_model_summary[site][model] = {}
        
        for metric in error_metrics:
            # Get error values for each fold
            values = [d[metric] for d in model_data]
            site_model_summary[site][model][f"{metric}_mean"] = np.mean(values)
            site_model_summary[site][model][f"{metric}_std"] = np.std(values)

In [ ]:
# Plot mean and standard deviation of errors across sites
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Use RMSSE as the primary metric for plotting
metric = "root_mean_squared_scaled_error"

# Prepare data for plotting
sites = UCI_CLIENT_SITES_TO_VALIDATE
x_pos = np.arange(len(sites))

# Color map for models
colors = plt.cm.tab10(np.linspace(0, 1, len(MODELS)))
model_colors = dict(zip(MODELS, colors))

# Plot 1: Mean errors
for i, model in enumerate(MODELS):
    mean_values = [site_model_summary[site][model][f"{metric}_mean"] for site in sites]
    ax1.plot(
        x_pos, mean_values, marker='o', linewidth=2, markersize=8,
        label=model.upper(), color=model_colors[model]
    )

ax1.set(ylabel='Mean RMSSE', title='Mean Error Across Sites')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(sites, rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Standard deviation of errors
for i, model in enumerate(MODELS):
    std_values = [site_model_summary[site][model][f"{metric}_std"] for site in sites]
    ax2.plot(
        x_pos, std_values, marker='o', linewidth=2, markersize=8,
        label=model.upper(), color=model_colors[model]
    )

ax2.set(ylabel='Standard Deviation of RMSSE', title='Error Variability Across Sites')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(sites, rotation=45, ha='right')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_OUTPUT_DIR / "summary_forecast_errors.png", dpi=300);
plt.show()

In [ ]:
# Create markdown table with summary results
from IPython.display import Markdown, display

# Calculate overall averages across all sites for each model and metric
overall_averages = {}
for model in MODELS:
    overall_averages[model] = {}
    for metric in error_metrics:
        # Calculate average of means across all sites
        mean_values = [site_model_summary[site][model][f"{metric}_mean"] for site in UCI_CLIENT_SITES_TO_VALIDATE]
        std_values = [site_model_summary[site][model][f"{metric}_std"] for site in UCI_CLIENT_SITES_TO_VALIDATE]
        
        overall_averages[model][f"{metric}_mean_avg"] = np.mean(mean_values)
        overall_averages[model][f"{metric}_std_avg"] = np.mean(std_values)

# Create markdown table for each metric
for metric in error_metrics:
    metric_display_name = metric.replace("_", " ").title()
    
    table_md = f"## {metric_display_name} Summary\n\n"
    table_md += "| Model | " + " | ".join([f"{site} Mean" for site in UCI_CLIENT_SITES_TO_VALIDATE])
    table_md += " | " + " | ".join([f"{site} Std" for site in UCI_CLIENT_SITES_TO_VALIDATE])
    table_md += " | Overall Mean Avg | Overall Std Avg |\n"
    
    # Header separator
    table_md += "|" + "---|" * (len(UCI_CLIENT_SITES_TO_VALIDATE) * 2 + 3) + "\n"
    
    # Data rows
    for model in MODELS:
        table_md += f"| **{model.upper()}** |"
        
        # Add mean values for each site
        for site in UCI_CLIENT_SITES_TO_VALIDATE:
            mean_val = site_model_summary[site][model][f"{metric}_mean"]
            table_md += f" {mean_val:.4f} |"
        
        # Add std values for each site  
        for site in UCI_CLIENT_SITES_TO_VALIDATE:
            std_val = site_model_summary[site][model][f"{metric}_std"]
            table_md += f" {std_val:.4f} |"
        
        # Add overall averages
        mean_avg = overall_averages[model][f"{metric}_mean_avg"]
        std_avg = overall_averages[model][f"{metric}_std_avg"]
        table_md += f" **{mean_avg:.4f}** | **{std_avg:.4f}** |\n"
    
    display(Markdown(table_md))
    print("\n")